# DOA Performance Evaluation with Multiple Algorithms

This notebook demonstrates the use of our custom DOA performance evaluator to analyze the performance of multiple DOA algorithms under different conditions:
1. Performance under different SNRs (using MSE and Bias metrics)
2. Performance under different snapshot numbers (using MSE metric)
3. Resolution performance under different angular separations (using MSE metric)

We evaluate the following DOA algorithms:
- MUSIC
- Root-MUSIC
- ESPRIT
- MVDR
- Bartlett
- MinNorm

In [ ]:
import numpy as np
import doatools.model as model
import doatools.estimation as estimation
import doatools.performance as perf
import matplotlib.pyplot as plt
from tqdm import tqdm
%matplotlib inline

## 1. Performance under different SNRs

We evaluate the performance of multiple DOA algorithms under different SNRs using MSE and Bias metrics.

In [ ]:
# Parameters setup
wavelength = 1.0  # normalized
d0 = wavelength / 2

# Create a 12-element ULA
array = model.UniformLinearArray(12, d0)

# Place 5 sources uniformly within (-pi/4, pi/4)
sources = model.FarField1DSourcePlacement(
    np.linspace(-np.pi/4, np.pi/4, 5)
)

# Set up multiple estimators
estimators = {
    'RootMUSIC': estimation.RootMUSIC1D(wavelength),
    'MUSIC': estimation.MUSIC(array, wavelength, estimation.FarField1DSearchGrid(
        start=-np.pi/2, stop=np.pi/2, size=1000, unit='rad'
    )),
    'ESPRIT': estimation.Esprit1D(wavelength),
    'MVDR': estimation.MVDRBeamformer(array, wavelength, estimation.FarField1DSearchGrid(
        start=-np.pi/2, stop=np.pi/2, size=1000, unit='rad'
    )),
    'Bartlett': estimation.BartlettBeamformer(array, wavelength, estimation.FarField1DSearchGrid(
        start=-np.pi/2, stop=np.pi/2, size=1000, unit='rad'
    )),
    'MinNorm': estimation.MinNorm(array, wavelength, estimation.FarField1DSearchGrid(
        start=-np.pi/2, stop=np.pi/2, size=1000, unit='rad'
    ))
}

# Test parameters
snrs = np.linspace(-20, 10, 10)  # SNR values from -20 dB to 10 dB
n_snapshots = 200
n_monte_carlo = 1000  # Reduced for faster demonstration

print(f"Evaluating {len(snrs)} SNR values with {len(estimators)} estimators...")
print(f"Algorithms: {', '.join(estimators.keys())}")

# Store results
all_results = {}
for est_name in estimators.keys():
    all_results[est_name] = {
        'mse': [],
        'bias': []
    }
sto_crb_results = []

for snr in tqdm(snrs):
    # Use our custom evaluator
    result = perf.evaluate_performance(
        array=array,
        sources=sources,
        snr=snr,
        n_snapshots=n_snapshots,
        n_monte_carlo=n_monte_carlo,
        estimators=estimators,
        crb_types=['sto'],
        metrics=['mse', 'bias']
    )
    
    # Extract results
    for est_name in estimators.keys():
        all_results[est_name]['mse'].append(result.estimator_results[est_name]['mse'])
        all_results[est_name]['bias'].append(result.estimator_results[est_name]['bias'])
    sto_crb_results.append(result.crb_values['sto'])

In [ ]:
# Plot MSE vs SNR for all estimators
plt.figure(figsize=(12, 6))

# Plot CRB
plt.semilogy(snrs, sto_crb_results, '--k', linewidth=2, label='Stochastic CRB')

# Plot each estimator's MSE
colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'orange']
for i, (est_name, results) in enumerate(all_results.items()):
    plt.semilogy(snrs, results['mse'], '-x', color=colors[i % len(colors)], label=est_name)

plt.xlabel('SNR (dB)')
plt.ylabel(r'MSE (rad$^2$)')
plt.grid(True, which='both', linestyle='--')
plt.legend(loc='lower left')
plt.title('MSE vs. SNR for Multiple DOA Algorithms')
plt.margins(x=0)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Bias vs SNR for all estimators
plt.figure(figsize=(12, 6))

# Plot each estimator's Bias
colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'orange']
for i, (est_name, results) in enumerate(all_results.items()):
    plt.plot(snrs, results['bias'], '-x', color=colors[i % len(colors)], label=est_name)

plt.xlabel('SNR (dB)')
plt.ylabel('Bias (rad)')
plt.grid(True, linestyle='--')
plt.legend(loc='upper right')
plt.title('Bias vs. SNR for Multiple DOA Algorithms')
plt.margins(x=0)
plt.tight_layout()
plt.show()

## 2. Performance under different snapshot numbers

We evaluate the performance of multiple DOA algorithms under different snapshot numbers using MSE metric.

In [ ]:
# Test parameters
snr = 0.0  # dB
snapshots = [10, 20, 50, 100, 200, 500]  # Different snapshot numbers
n_monte_carlo = 1000  # Reduced for faster demonstration

print(f"Evaluating {len(snapshots)} snapshot values with {len(estimators)} estimators...")

# Store results
snapshot_results = {}
for est_name in estimators.keys():
    snapshot_results[est_name] = {
        'mse': []
    }

for n_snap in tqdm(snapshots):
    # Use our custom evaluator
    result = perf.evaluate_performance(
        array=array,
        sources=sources,
        snr=snr,
        n_snapshots=n_snap,
        n_monte_carlo=n_monte_carlo,
        estimators=estimators,
        crb_types=['sto'],
        metrics=['mse']
    )
    
    # Extract results
    for est_name in estimators.keys():
        snapshot_results[est_name]['mse'].append(result.estimator_results[est_name]['mse'])

In [ ]:
# Plot MSE vs Snapshots for all estimators
plt.figure(figsize=(12, 6))

# Plot each estimator's MSE
colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'orange']
for i, (est_name, results) in enumerate(snapshot_results.items()):
    plt.semilogy(snapshots, results['mse'], '-x', color=colors[i % len(colors)], label=est_name)

plt.xlabel('Number of Snapshots')
plt.ylabel(r'MSE (rad$^2$)')
plt.grid(True, which='both', linestyle='--')
plt.legend(loc='lower left')
plt.title('MSE vs. Snapshots for Multiple DOA Algorithms')
plt.margins(x=0)
plt.tight_layout()
plt.show()

## 3. Resolution performance under different angular separations

We evaluate the resolution performance of multiple DOA algorithms under different angular separations using MSE metric.

In [ ]:
# Parameters setup for resolution test
# Create a 10-element ULA for better resolution comparison
array_res = model.UniformLinearArray(10, d0)

# Set up estimators for resolution test
estimators_res = {
    'RootMUSIC': estimation.RootMUSIC1D(wavelength),
    'MUSIC': estimation.MUSIC(array_res, wavelength, estimation.FarField1DSearchGrid(
        start=-np.pi/2, stop=np.pi/2, size=1000, unit='rad'
    )),
    'ESPRIT': estimation.Esprit1D(wavelength),
    'MVDR': estimation.MVDRBeamformer(array_res, wavelength, estimation.FarField1DSearchGrid(
        start=-np.pi/2, stop=np.pi/2, size=1000, unit='rad'
    )),
    'Bartlett': estimation.BartlettBeamformer(array_res, wavelength, estimation.FarField1DSearchGrid(
        start=-np.pi/2, stop=np.pi/2, size=1000, unit='rad'
    ))
}

# Test parameters
snr = 0.0  # dB
n_snapshots = 100
n_monte_carlo = 1000  # Reduced for faster demonstration

# Angular separations in degrees, converted to radians
delta_thetas_deg = np.linspace(0.5, 10.0, 10)  # From 0.5° to 10°
delta_thetas_rad = np.deg2rad(delta_thetas_deg)

print(f"Evaluating {len(delta_thetas_deg)} angular separations with {len(estimators_res)} estimators...")

# Store results
resolution_results = {}
for est_name in estimators_res.keys():
    resolution_results[est_name] = {
        'mse': []
    }

for delta_theta in tqdm(delta_thetas_rad):
    # Create two sources with the current separation
    sources_res = model.FarField1DSourcePlacement(
        [-delta_theta / 2, delta_theta / 2]
    )
    
    # Use our custom evaluator
    result = perf.evaluate_performance(
        array=array_res,
        sources=sources_res,
        snr=snr,
        n_snapshots=n_snapshots,
        n_monte_carlo=n_monte_carlo,
        estimators=estimators_res,
        crb_types=['sto'],
        metrics=['mse']
    )
    
    # Extract results
    for est_name in estimators_res.keys():
        resolution_results[est_name]['mse'].append(result.estimator_results[est_name]['mse'])

In [ ]:
# Plot MSE vs Angular Separation for all estimators
plt.figure(figsize=(12, 6))

# Plot each estimator's MSE
colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'orange']
for i, (est_name, results) in enumerate(resolution_results.items()):
    plt.semilogy(delta_thetas_deg, results['mse'], '-x', color=colors[i % len(colors)], label=est_name)

plt.xlabel('Angular Separation (degrees)')
plt.ylabel(r'MSE (rad$^2$)')
plt.grid(True, which='both', linestyle='--')
plt.legend(loc='upper right')
plt.title('Resolution Performance vs. Angular Separation')
plt.margins(x=0)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the use of our custom DOA performance evaluator to analyze multiple DOA algorithms under various conditions. Here are the key findings:

1. **SNR Performance**:
   - All algorithms show improved MSE as SNR increases
   - Root-MUSIC and ESPRIT generally perform closest to the CRB
   - Bartlett and MVDR typically have higher MSE than subspace-based methods
   - Bias decreases with increasing SNR for all algorithms

2. **Snapshot Performance**:
   - Increasing the number of snapshots improves MSE for all algorithms
   - The improvement is more significant for subspace-based methods at lower snapshot counts
   - Root-MUSIC and ESPRIT maintain good performance even with fewer snapshots

3. **Resolution Performance**:
   - All algorithms show improved MSE as angular separation increases
   - Root-MUSIC and ESPRIT outperform other algorithms for smaller separations
   - Bartlett has the lowest resolution capability
   - MUSIC performs well but is computationally more expensive than Root-MUSIC

Our custom evaluator provides a flexible and efficient way to evaluate and compare multiple DOA algorithms under various conditions, supporting multiple metrics and scenarios.